# Offline policy training & evaluation

Inspiration from [this turorial](https://vowpalwabbit.org/docs/vowpal_wabbit/python/latest/tutorials/off_policy_evaluation.html). We use an **incrementally trained** VW policy with adf-format data.


In [1]:
import os
import random

import vowpalwabbit

In [2]:
if os.path.isfile("candidate_model.vw"):
    os.remove("candidate_model.vw")

random.seed()

In [7]:
def get_examples(filename):
    examples = []
    with open(filename, "r") as fin:
        example = []
        for line in fin:
            if line.startswith("shared |"):
                if len(example) > 0:
                    examples.append(example)
                example = []
                example.append(line.strip("\n"))
            else:
                if len(line.strip("\n")) > 0:
                    example.append(line.strip("\n"))
        examples.append(example)
    return examples


def dialogue_length(line):
    return len(line.split("|")[1].strip().split(" "))


def calculate_dialogue_lengths(examples):
    lengths = {}
    for e in examples:
        length = dialogue_length(e[0])
        if length in lengths:
            lengths[length] += 1
        else:
            lengths[length] = 1
    return lengths


def get_average(lengths: dict):
    sum = 0
    count = 0
    for k, v in lengths.items():
        sum += k * v
        count += v
    return sum / count, count


def sample_custom_pmf(pmf):
    total = sum(pmf)
    scale = 1 / total
    pmf = [x * scale for x in pmf]
    draw = random.random()
    sum_prob = 0.0
    for index, prob in enumerate(pmf):
        sum_prob += prob
        if sum_prob > draw:
            return index, prob

In [11]:
examples_new = get_examples("applications/mock_dialogues_service/new_data.dat")
examples_old = get_examples("data/data.dat")

random.shuffle(examples_old)
random.shuffle(examples_new)

data = examples_old
random.shuffle(data)

data_train = data[: round(0.8 * len(data))]
data_validate = data[round(0.8 * len(data)) :]
# data_train = examples_old
# data_validate = examples_new

In [ ]:
import matplotlib.pyplot as plt

old = calculate_dialogue_lengths(examples_old)
old_average, old_count = get_average(old)
new = calculate_dialogue_lengths(examples_new)
new_average, new_count = get_average(new)

fig, (ax1, ax2) = plt.subplots(
    1,
    2,
    sharey=True,
    sharex=True,
    figsize=(10, 5),
)
fig.suptitle("Comparison of dialogue lengths")


ax1.bar(
    list(old.keys()),
    old.values(),
    color="g",
)
ax1.set(xlabel="Dialogue length")
ax1.set_title(f"Old data, avg. len: {old_average:,.2f}, total: {old_count} ")
ax2.bar(list(new.keys()), new.values(), color="g")
ax2.set_title(f"New data, avg. len: {new_average:,.2f}, total: {new_count}")
ax2.set(xlabel="Dialogue length")

plt.show()

Train the model.


In [12]:
cb_type = "ips"

vw_train = vowpalwabbit.Workspace(
    cb_explore_adf=True,
    cb_type=cb_type,
    f="candidate_model.vw",
)

final_regressor = candidate_model.vw
using no cache
Reading datafile = none
num sources = 0
Num weight bits = 18
learning rate = 0.5
initial_t = 0
power_t = 0.5
cb_type = ips
Enabled learners: gd, scorer-identity, csoaa_ldf-rank, cb_adf, cb_explore_adf_greedy, shared_feature_merger
Input label = CB
Output pred = ACTION_PROBS
average  since         example        example        current        current  current
loss     last          counter         weight          label        predict features


In [13]:
for i in range(len(data_train)):
    vw_format = vw_train.parse(data_train[i])
    vw_train.learn(vw_format)
    vw_train.finish_example(vw_format)


finished run
number of examples = 1
weighted example sum = 1.000000
weighted label sum = 0.000000
average loss = n.a.
total feature number = 46
1.000000 1.000000            1            1.0        0:1:0.1          0:0.1       30
0.500000 0.000000            2            2.0      0:0:0.091         1:0.95       33
0.262500 0.025000            4            4.0        0:1:0.1         1:0.95       30
0.150000 0.037500            8            8.0       0:1:0.14         1:0.96       21
0.087500 0.025000           16           16.0      0:1:0.083         1:0.95       36
0.709375 1.331250           32           32.0      0:1:0.091         6:0.95       33
0.710156 0.710938           64           64.0        0:1:0.2         4:0.96       15
0.458594 0.207031          128          128.0       0:0:0.12         6:0.96       24
0.391504 0.324414          256          256.0        0:1:0.1         7:0.95       30
0.332802 0.274099          512          512.0          0:0:1            0:1        3


In [14]:
vw_train.finish()


finished run
number of examples = 834
weighted example sum = 834.000000
weighted label sum = 0.000000
average loss = 0.311227
total feature number = 23193


In [15]:
vw_test = vowpalwabbit.Workspace(
    cb_explore_adf=True, cb_type=cb_type, i="candidate_model.vw", t=True
)

for i in range(len(data_validate)):
    vw_format = vw_train.parse(data_validate[i])
    vw_test.learn(vw_format)
    vw_test.finish_example(vw_format)
vw_test.finish()

only testing
using no cache
Reading datafile = none
num sources = 0
Num weight bits = 18
learning rate = 0.5
initial_t = 834
power_t = 0.5
cb_type = ips
Enabled learners: gd, scorer-identity, csoaa_ldf-rank, cb_adf, cb_explore_adf_greedy, shared_feature_merger
Input label = CB
Output pred = ACTION_PROBS
average  since         example        example        current        current  current
loss     last          counter         weight          label        predict features
0.000000 0.000000            1            1.0       0:0:0.25         0:0.25       12
0.000000 0.000000            2            2.0       0:0:0.12         0:0.12       24
0.000000 0.000000            4            4.0      0:0:0.083         0:0.08       36
0.250000 0.500000            8            8.0      0:1:0.091         0:0.09       33
0.187500 0.125000           16           16.0        0:0:0.2          0:0.2       15
0.406250 0.625000           32           32.0       0:0:0.11         0:0.11       27
0.437500 0.4687

In [5]:
sample = vw_train.predict("""shared | vaccines covid_stats 
| no_action vaccines covid_stats 
| icu_stats_form 
| vaccine_stats_form
| pharmacy_form
| ill_questionnaire_form
| ways_of_protection
| symptoms
| test_types
| test_cost
| what_to_do_if_positive
| underlying_conditions""")

print(sample)
print(sample_custom_pmf(sample))

[0.09090909361839294, 0.09090909361839294, 0.09090909361839294, 0.09090909361839294, 0.09090909361839294, 0.09090909361839294, 0.09090909361839294, 0.09090909361839294, 0.09090909361839294, 0.09090909361839294, 0.09090909361839294]


n.a.     n.a.                1            1.0        unknown         0:0.09       46


NameError: name 'sample_custom_pmf' is not defined

Testing


|                      | 0        | 100      | 200      | 400      | All      |
| -------------------- | -------- | -------- | -------- | -------- | -------- |
| number of examples   | 1043     | 1143     | 1243     | 1443     | 1625     |
| average loss         | 0.330858 | 0.356043 | 0.363716 | 0.374223 | 0.407437 |
| total feature number | 48366    | 54194    | 60202    | 71476    | 82085    |


Useful sources:

- https://stackoverflow.com/questions/66996884/confusion-with-vowpal-wabbit-contextual-bandit-training-data-formatting
- https://stackoverflow.com/questions/28640837/vowpal-wabbit-how-to-represent-categorical-features/28643968#28643968
